# 用 llama-server 跑 Qwen3.8-27B-Uncensored (Q8_0)

需要 **GPU 运行时**（G4 / A100 / H100）。依次运行：
1. 挂载 Drive，把模型复制到本机硬盘
2. 编译 llama.cpp（首次约 5–10 分钟，之后从 Drive 复用）
3. 启动服务（端口 8000，带 API key）
4. 在 Colab 里对话 + 测速
5. **建立隧道** → `你在 secrets.json 里配置的域名`，本地 Mac / Magpie 从这里调用
6. 看日志 / 用完关机

In [ ]:
# 1. 挂载 Drive，复制模型到 Colab 机器硬盘（从本机硬盘加载比从 Drive 快得多）
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/models/JonathanColetti__Qwen3.8-27B-Uncensored-GGUF"
FILE = "Qwen3.8-27B-Uncensored-Q8_0.gguf"
MODEL_PATH = f"/content/models/{FILE}"
BIN_DIR = "/content/drive/MyDrive/models/bin"   # 编译好的 llama-server 存这里

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!mkdir -p /content/models
!rsync -ah --progress "{DRIVE_DIR}/{FILE}" /content/models/
!ls -lh /content/models

In [ ]:
# 2. 编译 llama.cpp (CUDA)。如果 Drive 里已有同架构的编译结果就直接复用
import os, subprocess
cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0]
ARCH = cap.replace(".", "")          # H100 -> 90, A100 -> 80
CACHED = f"{BIN_DIR}/llama-server-sm{ARCH}"
SERVER = "/content/llama-server"

if os.path.exists(CACHED):
    print("复用 Drive 里的编译结果:", CACHED)
    !cp "{CACHED}" {SERVER} && chmod +x {SERVER}
else:
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build \
        -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={ARCH} \
        -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
    !cmake --build /content/llama.cpp/build --target llama-server -j $(nproc) 2>&1 | tail -3
    !cp /content/llama.cpp/build/bin/llama-server {SERVER}
    !mkdir -p "{BIN_DIR}" && cp {SERVER} "{CACHED}"
    print("编译完成并已存到 Drive:", CACHED)

!{SERVER} --version

In [ ]:
# 3. 后台启动 llama-server（带 API key，只监听本机，外网走第 5 步的隧道）
import time, urllib.request, secrets, json

def get_secret(name):
    """先读 Colab Secrets（左侧钥匙图标），再读 Drive 里的 MyDrive/models/secrets.json"""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    try:
        return json.load(open("/content/drive/MyDrive/models/secrets.json")).get(name)
    except Exception:
        return None

import os
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"找不到 {MODEL_PATH}：新运行时的硬盘是空的，请先运行第 1 步复制模型")
subprocess.run(["pkill", "-f", "llama-server"]); time.sleep(2)   # 如果之前启动过，先停掉旧的

vram_gb = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).decode().split()[0]) / 1024
if vram_gb >= 70:      # H100 80GB / G4 96GB
    CTX, KV = 32768, []
else:                  # A100 40GB：上下文短一些，KV cache 量化省显存
    CTX, KV = 16384, ["-ctk", "q8_0", "-ctv", "q8_0"]
print(f"显存 {vram_gb:.0f}GB -> 上下文 {CTX}")

API_KEY = get_secret("LLM_API_KEY") or secrets.token_urlsafe(24)   # 配了固定 key 就用固定的
print("API key:", "固定（来自 secrets）" if get_secret("LLM_API_KEY") else "本次随机生成")
cmd = [SERVER, "-m", MODEL_PATH, "--alias", "qwen3.8-27b", "--api-key", API_KEY,
       "-ngl", "99", "-c", str(CTX), *KV,
       "--spec-type", "draft-mtp", "--spec-draft-n-max", "1",   # README 实测 n_max=1 最快
       "--jinja", "--host", "127.0.0.1", "--port", "8000"]
log = open("/content/llama-server.log", "w")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

for i in range(180):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8000/health").status == 200:
            print(f"服务已就绪（{i*2}s）"); break
    except Exception:
        pass
    if proc.poll() is not None:
        print("启动失败，日志："); break
    time.sleep(2)
!tail -n 15 /content/llama-server.log
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# 4. 对话 + 测速（流式）
!pip install -q openai
from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key=API_KEY)

def chat(prompt, max_tokens=1024):
    t0, first, n = time.time(), None, 0
    stream = client.chat.completions.create(
        model="qwen3.8-27b", stream=True, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
        stream_options={"include_usage": True})
    for chunk in stream:
        if chunk.usage: n = chunk.usage.completion_tokens
        if not chunk.choices: continue
        d = chunk.choices[0].delta
        text = getattr(d, "reasoning_content", None) or d.content or ""
        if text and first is None: first = time.time()
        print(text, end="", flush=True)
    dt = time.time() - (first or t0)
    print(f"\n\n--- 首字 {(first or t0)-t0:.2f}s | {n} tokens | {n/dt:.1f} tok/s ---")

chat("用三句话介绍一下你自己。")

In [ ]:
# 5. 建立 Cloudflare 隧道 （读取 Drive 里的 models/secrets.json，用 scripts/setup_tunnel.sh 生成）
#    没配 secrets 时自动退回临时 trycloudflare 地址
import re
subprocess.run(["pkill", "-f", "cloudflared"])   # 停掉旧隧道（用 Python 调，避免 pkill 误杀自己所在的 shell）
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
tlog = open("/content/cloudflared.log", "w")
CF_TOKEN, CF_HOST = get_secret("CF_TUNNEL_TOKEN"), get_secret("CF_HOSTNAME")

if CF_TOKEN and CF_HOST:
    tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate",
                               "--url", "http://127.0.0.1:8000", "run", "--token", CF_TOKEN],
                              stdout=tlog, stderr=subprocess.STDOUT)
    PUBLIC_URL = f"https://{CF_HOST}"
    for _ in range(30):
        if "Registered tunnel connection" in open("/content/cloudflared.log").read(): break
        time.sleep(1)
    else:
        print("固定隧道可能没连上，日志："); PUBLIC_URL = None
        !tail -n 20 /content/cloudflared.log
else:
    print("⚠️ 没读到 secrets.json，使用临时地址")
    tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate",
                               "--url", "http://127.0.0.1:8000"], stdout=tlog, stderr=subprocess.STDOUT)
    PUBLIC_URL = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m: PUBLIC_URL = m.group(0); break
        time.sleep(1)
    if not PUBLIC_URL:
        print("隧道没起来，日志：")
        !tail -n 20 /content/cloudflared.log

if PUBLIC_URL:
    fixed = bool(CF_TOKEN and CF_HOST and get_secret("LLM_API_KEY"))
    print("✅ 固定地址 + 固定 key，本地 .env 不用改\n" if fixed
          else "把下面三行覆盖写入 Mac 上的 ~/探索/colab/.env ：\n")
    print(f"COLAB_LLM_URL={PUBLIC_URL}/v1")
    if not fixed: print(f"COLAB_LLM_KEY={API_KEY}")
    print("COLAB_LLM_MODEL=qwen3.8-27b")
    # 自检：从公网地址访问一次
    time.sleep(5)
    try:
        r = urllib.request.urlopen(urllib.request.Request(PUBLIC_URL + "/v1/models",
                headers={"Authorization": "Bearer " + API_KEY,
                         "User-Agent": "colab-llm/1.0"}), timeout=20)
        print("公网自检:", r.status, json.loads(r.read())["data"][0]["id"])
    except Exception as e:
        print("公网自检失败（可再等几十秒重试第 5 步）:", e)

In [ ]:
# 6.（可选）看日志
!tail -n 30 /content/llama-server.log
!tail -n 10 /content/cloudflared.log

# 用完关机：取消下面两行注释并运行，立刻释放 GPU，停止消耗计算单元
# from google.colab import runtime
# runtime.unassign()